# DESPLIEGUE DEL MODELO PREDICTIVO CON INTERFAZ GRÁFICA
## Predicción de Accidente Cerebrovascular (ACV) — Punto 1.D

**Realizado por:** Jacobo Arévalo Zea · Zharick Rocío Carrillo Grijalba · Nicole Yuqui Vásquez — Universidad Pontificia Bolivariana

**Pasos del despliegue:**
1. Cargar el modelo final (`modelo_final.pkl`, generado en el notebook `Mineria_Datos_Python.ipynb`).
2. Cargar los datos futuros (`datos_futuros.csv`).
3. Preparar los datos futuros: mismas columnas (dummies) y en el mismo orden que el entrenamiento.
4. Aplicar el modelo para la predicción.
5. Publicar una **interfaz gráfica (Streamlit)** para que cualquier persona pueda usar el modelo.

**Nota:** el archivo `.pkl` guarda el **pipeline completo** (SMOTENC + MinMaxScaler + Random Forest). Por eso aquí **no se escala a mano**:
el pipeline aplica internamente el mismo escalado del entrenamiento. Así se evita el error de la práctica anterior, donde los datos
se escalaban por fuera y el modelo había sido entrenado sin escalar, lo que hacía que siempre predijera 0.

## 0. Librerías

In [ ]:
!pip install -q streamlit

import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

## 1. Cargar el modelo

Se sube a Colab el archivo `modelo_final.pkl` generado en el notebook de modelos (panel de archivos 📁 → subir).

Si el archivo no está, la celda **regenera exactamente el mismo modelo final** con el mismo código del notebook de modelos
(mismos datos, mismas variables, mismo pipeline y los hiperparámetros ganadores del GridSearch, `random_state=42`).
Así el `.pkl` siempre queda creado con la misma versión de scikit-learn que se está usando y no hay errores de compatibilidad.

In [ ]:
def regenerar_modelo_final():
    """Mismo procedimiento del notebook Mineria_Datos_Python.ipynb (Parte 1 a Parte 3)."""
    from sklearn.impute import SimpleImputer
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.ensemble import RandomForestClassifier
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTENC

    try:
        import kagglehub
        path = kagglehub.dataset_download("aouatifcherdid/healthcare-dataset-stroke-data")
        datos1 = pd.read_csv(os.path.join(path, "healthcare-dataset-stroke-data.csv"))
    except Exception:
        datos1 = pd.read_csv("https://raw.githubusercontent.com/fenago/datasets/main/healthcare-dataset-stroke-data.csv")

    # Limpieza y selección de factores (igual que en el notebook de modelos)
    datos1 = datos1.drop(["id", "gender", "Residence_type"], axis=1)
    datos1[["bmi"]] = SimpleImputer(strategy="median").fit_transform(datos1[["bmi"]])
    d = pd.get_dummies(datos1, columns=["ever_married"], drop_first=True, dtype=int)
    d = pd.get_dummies(d, columns=["work_type", "smoking_status"], drop_first=False, dtype=int)
    X, y = d.drop("stroke", axis=1), d["stroke"]

    numericas = ["age", "avg_glucose_level", "bmi"]
    posiciones_cat = [i for i, c in enumerate(X.columns) if c not in numericas]
    modelo = Pipeline([
        ("balanceo", SMOTENC(categorical_features=posiciones_cat, k_neighbors=2, random_state=42)),
        ("escalado", ColumnTransformer([("num", MinMaxScaler(), numericas)], remainder="passthrough")),
        ("modelo", RandomForestClassifier(n_estimators=150, max_samples=0.9, criterion="gini",
                                          max_depth=20, min_samples_leaf=10, random_state=42)),
    ])
    modelo.fit(X, y)   # 100 % de los datos, como el modelo final
    with open("modelo_final.pkl", "wb") as f:
        pickle.dump([modelo, X.columns.tolist()], f)


if not os.path.exists("modelo_final.pkl"):
    print("No se encontró modelo_final.pkl -> se regenera con el código del notebook de modelos...")
    regenerar_modelo_final()

with open("modelo_final.pkl", "rb") as f:
    modelo, variables = pickle.load(f)

print("Modelo cargado:", type(modelo.named_steps["modelo"]).__name__)
print("Pasos del pipeline:", list(modelo.named_steps))
print("Variables que espera el modelo:", variables)

## 2. Cargar los datos futuros

Se sube `datos_futuros.csv` al panel de archivos. Si no está, se crea con los mismos 5 pacientes del archivo original.

In [ ]:
# Si datos_futuros.csv no se subió al panel de archivos, se crea con los mismos 5 pacientes del archivo original
if not os.path.exists("datos_futuros.csv"):
    columnas = ["age", "hypertension", "heart_disease", "ever_married_Yes",
                "work_type_Govt_job", "work_type_Never_worked", "work_type_Private",
                "work_type_Self-employed", "work_type_children", "avg_glucose_level", "bmi",
                "smoking_status_Unknown", "smoking_status_formerly smoked",
                "smoking_status_never smoked", "smoking_status_smokes"]
    pacientes = [
        [67, 1, 1, 1, 0, 0, 1, 0, 0, 210.5, 32.1, 0, 1, 0, 0],
        [45, 0, 0, 1, 0, 0, 0, 1, 0,  95.3, 27.4, 0, 0, 1, 0],
        [23, 0, 0, 0, 0, 0, 1, 0, 0,  82.7, 21.8, 1, 0, 0, 0],
        [58, 1, 0, 1, 1, 0, 0, 0, 0, 145.0, 29.9, 0, 0, 0, 1],
        [ 8, 0, 0, 0, 0, 1, 0, 0, 1,  70.2, 17.5, 1, 0, 0, 0],
    ]
    pd.DataFrame(pacientes, columns=columnas).to_csv("datos_futuros.csv", index=False)

data = pd.read_csv("datos_futuros.csv")
data

## 3. Preparación de los datos futuros

- **Revisión de consistencia:** cada paciente debe tener **una sola** categoría activa en `work_type` y en `smoking_status`.
  La fila 4 (niño de 8 años) tiene `work_type_Never_worked = 1` y `work_type_children = 1` al mismo tiempo.
  Como es menor de edad, se deja solo `children`.
- **Mismas columnas y mismo orden** del entrenamiento con `reindex` (si falta alguna dummy se rellena con 0).
- **No se normaliza aquí:** el pipeline del modelo ya trae el MinMaxScaler ajustado en el entrenamiento.

In [ ]:
def preparar(df):
    df = df.copy()
    if "Edad" in df.columns and "age" not in df.columns:
        df = df.rename(columns={"Edad": "age"})
    df = df.reindex(columns=variables, fill_value=0)

    avisos = []
    trabajo = [c for c in variables if c.startswith("work_type_")]
    fuma = [c for c in variables if c.startswith("smoking_status_")]
    for i, fila in df.iterrows():
        if df.loc[i, trabajo].sum() > 1:
            activas = [c for c in trabajo if fila[c] == 1]
            if fila["work_type_children"] == 1 and fila["age"] < 18:
                df.loc[i, trabajo] = 0
                df.loc[i, "work_type_children"] = 1
                avisos.append(f"Fila {i}: tenía {activas}; se deja work_type_children (edad {fila['age']}).")
            else:
                avisos.append(f"Fila {i}: tiene varias categorías de trabajo activas {activas}. Revisar.")
        if df.loc[i, fuma].sum() != 1:
            avisos.append(f"Fila {i}: smoking_status debe tener exactamente una categoría activa.")
    return df, avisos

data_preparada, avisos = preparar(data)
for a in avisos:
    print("⚠️", a)
data_preparada

## 4. Aplicar el modelo para la predicción

In [ ]:
UMBRAL = 0.5

def nivel_riesgo(p):
    if p >= UMBRAL:
        return "ALTO"
    if p >= 0.25:
        return "MODERADO"
    return "BAJO"

probabilidades = modelo.predict_proba(data_preparada)[:, 1]
data["Probabilidad_ACV"] = probabilidades.round(3)
data["Prediccion"] = modelo.predict(data_preparada)
data["Riesgo"] = [nivel_riesgo(p) for p in probabilidades]
data[["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "Probabilidad_ACV", "Prediccion", "Riesgo"]]

**Interpretación:** 
- **Paciente 0** (67 años, hipertensión, enfermedad cardíaca y glucosa de 210,5): probabilidad de **67,8 %**, riesgo **ALTO**. El modelo lo clasifica como posible ACV (predicción = 1).
- **Paciente 3** (58 años, hipertensión, fumador y glucosa de 145): probabilidad de **36,9 %**, riesgo **MODERADO**. No supera el umbral de 0,5, pero conviene hacerle seguimiento.
- **Pacientes 1, 2 y 4** (45, 23 y 8 años, sin hipertensión ni cardiopatía): probabilidad menor al 14 %, riesgo **BAJO**.
- Las predicciones son coherentes con el análisis: la **edad**, la **glucosa**, la **hipertensión** y la **enfermedad cardíaca** son los factores que más aumentan el riesgo.

## 5. Interfaz gráfica del despliegue (Streamlit)

La interfaz gráfica se construyó con **Streamlit** y es **la misma app** que está publicada en Streamlit Community Cloud desde el
repositorio público de GitHub (`app.py`):

- **App en línea:** https://prediccion-acv-upb.streamlit.app
- **Repositorio:** https://github.com/nikkiyuki/prediccion-acv

La app tiene dos pestañas:
- **Paciente individual:** formulario en español con las variables del modelo. Muestra la probabilidad de ACV, el nivel de riesgo
  (Bajo < 25 % ≤ Moderado < 50 % ≤ Alto) y una alerta.
- **Archivo CSV:** se sube un archivo con el formato de `datos_futuros.csv` (o se usa el del proyecto) y se descargan las predicciones.

Por dentro, la app convierte las respuestas del formulario a las mismas columnas dummies que espera el modelo y usa el
`modelo_final.pkl` cargado en el paso 1.

**5.1 Se crea el archivo `app.py`:**

In [ ]:
%%writefile app.py
"""Despliegue del modelo predictivo de ACV con Streamlit.

Minería de Datos en Python — Universidad Pontificia Bolivariana
Jacobo Arévalo Zea · Zharick Rocío Carrillo Grijalba · Nicole Yuqui Vásquez

En Colab se crea con %%writefile y se ejecuta en la celda siguiente.
"""
import os
import pickle

import pandas as pd
import streamlit as st

st.set_page_config(page_title="Predicción de ACV", page_icon="🧠", layout="wide")

UMBRAL = 0.5
TRABAJO = {"Empleado privado": "Private", "Independiente": "Self-employed", "Empleado público": "Govt_job",
           "Niño / menor de edad": "children", "Nunca ha trabajado": "Never_worked"}
FUMA = {"Nunca ha fumado": "never smoked", "Exfumador": "formerly smoked",
        "Fuma actualmente": "smokes", "Desconocido": "Unknown"}


@st.cache_resource(show_spinner="Cargando el modelo...")
def cargar_modelo():
    """Carga el pipeline completo (SMOTENC + MinMaxScaler + Random Forest) guardado en modelo_final.pkl."""
    with open("modelo_final.pkl", "rb") as f:
        modelo, variables = pickle.load(f)
    return modelo, variables


modelo, variables = cargar_modelo()


def nivel_riesgo(p):
    if p >= UMBRAL:
        return "ALTO"
    if p >= 0.25:
        return "MODERADO"
    return "BAJO"


def preparar(df):
    """Mismas columnas (dummies) y orden del entrenamiento + revisión de consistencia."""
    df = df.copy()
    if "Edad" in df.columns and "age" not in df.columns:
        df = df.rename(columns={"Edad": "age"})
    df = df.reindex(columns=variables, fill_value=0)
    avisos = []
    trabajo = [c for c in variables if c.startswith("work_type_")]
    for i, fila in df.iterrows():
        if df.loc[i, trabajo].sum() > 1:
            activas = [c for c in trabajo if fila[c] == 1]
            if fila["work_type_children"] == 1 and fila["age"] < 18:
                df.loc[i, trabajo] = 0
                df.loc[i, "work_type_children"] = 1
                avisos.append(f"Fila {i}: tenía {activas}; se dejó work_type_children (edad {fila['age']:.0f}).")
            else:
                avisos.append(f"Fila {i}: tiene varias categorías de trabajo activas {activas}.")
    return df, avisos


# ---------------- Encabezado ----------------
st.title("🧠 Predicción de riesgo de Accidente Cerebrovascular (ACV)")
st.caption("Minería de Datos en Python · Universidad Pontificia Bolivariana — "
           "Jacobo Arévalo Zea · Zharick Rocío Carrillo Grijalba · Nicole Yuqui Vásquez")

with st.sidebar:
    st.header("Sobre el modelo")
    st.markdown(
        "- **Modelo:** Random Forest optimizado con GridSearch\n"
        "- **Pipeline:** SMOTENC → MinMaxScaler → Random Forest\n"
        "- **Datos:** Healthcare Stroke Data (5.110 pacientes)\n"
        "- **Prueba (30 %):** AUC ≈ 0.83 · Recall ≈ 0.65\n"
        f"- **Umbral de decisión:** {UMBRAL:.0%}"
    )
    st.info("Herramienta académica de tamizaje. No reemplaza un diagnóstico médico.")

tab1, tab2 = st.tabs(["👤 Paciente individual", "📄 Archivo CSV (varios pacientes)"])

# ---------------- Pestaña 1: paciente individual ----------------
with tab1:
    col_form, col_res = st.columns([1, 1], gap="large")
    with col_form:
        with st.form("formulario"):
            edad = st.slider("Edad (años)", 0, 100, 67)
            c1, c2 = st.columns(2)
            glucosa = c1.number_input("Glucosa promedio (mg/dL)", min_value=40.0, max_value=300.0, value=210.5, step=0.1)
            imc = c2.number_input("Índice de masa corporal (IMC)", min_value=10.0, max_value=100.0, value=32.1, step=0.1)
            c3, c4, c5 = st.columns(3)
            hipertension = c3.radio("Hipertensión", ["No", "Sí"], index=1, horizontal=True)
            cardiaca = c4.radio("Enfermedad cardíaca", ["No", "Sí"], index=1, horizontal=True)
            casado = c5.radio("¿Alguna vez casado(a)?", ["No", "Sí"], index=1, horizontal=True)
            trabajo = st.selectbox("Tipo de trabajo", list(TRABAJO), index=0)
            fuma = st.selectbox("Tabaquismo", list(FUMA), index=1)
            enviado = st.form_submit_button("Predecir", type="primary", use_container_width=True)

    with col_res:
        if enviado:
            fila = dict.fromkeys(variables, 0)
            fila.update({"age": edad, "hypertension": int(hipertension == "Sí"), "heart_disease": int(cardiaca == "Sí"),
                         "avg_glucose_level": glucosa, "bmi": imc, "ever_married_Yes": int(casado == "Sí")})
            fila["work_type_" + TRABAJO[trabajo]] = 1
            fila["smoking_status_" + FUMA[fuma]] = 1
            paciente = pd.DataFrame([fila])[variables]

            p = float(modelo.predict_proba(paciente)[0, 1])
            nivel = nivel_riesgo(p)

            st.subheader("Resultado")
            m1, m2 = st.columns(2)
            m1.metric("Probabilidad de ACV", f"{p:.1%}")
            m2.metric("Nivel de riesgo", nivel)
            st.progress(p)
            if p >= UMBRAL:
                st.error(f"⚠️ **Riesgo {nivel}.** El modelo clasifica al paciente como **posible caso de ACV**. "
                         "Se recomienda valoración médica prioritaria.")
            elif nivel == "MODERADO":
                st.warning(f"🔶 **Riesgo {nivel}.** No supera el umbral, pero se recomienda seguimiento.")
            else:
                st.success(f"✅ **Riesgo {nivel}.** El modelo clasifica al paciente como **sin ACV**.")
        else:
            st.info("Completa los datos del paciente y presiona **Predecir**.")

# ---------------- Pestaña 2: archivo CSV ----------------
with tab2:
    st.markdown("Sube un archivo con el mismo formato de `datos_futuros.csv` (variables en dummies). "
                "Si no tienes uno, usa el botón para probar con los datos futuros del proyecto.")
    archivo = st.file_uploader("Archivo CSV", type="csv")
    usar_ejemplo = st.button("Usar datos_futuros.csv del proyecto")

    df = None
    if archivo is not None:
        df = pd.read_csv(archivo)
    elif usar_ejemplo:
        df = pd.read_csv("datos_futuros.csv")

    if df is not None:
        X, avisos = preparar(df)
        for a in avisos:
            st.warning("⚠️ " + a)
        proba = modelo.predict_proba(X)[:, 1]
        salida = df.copy()
        salida["Probabilidad_ACV"] = proba.round(3)
        salida["Prediccion"] = (proba >= UMBRAL).astype(int)
        salida["Riesgo"] = [nivel_riesgo(p) for p in proba]
        st.dataframe(salida[["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi",
                             "Probabilidad_ACV", "Prediccion", "Riesgo"]], use_container_width=True)
        st.download_button("⬇️ Descargar predicciones", salida.to_csv(index=False).encode("utf-8"),
                           file_name="predicciones.csv", mime="text/csv")


In [ ]:
# 5.2 Se ejecuta la app de Streamlit en segundo plano y se muestra aquí mismo en Colab
import socket, subprocess, time
from google.colab import output

def puerto_libre(puerto):
    with socket.socket() as s:
        return s.connect_ex(("localhost", puerto)) != 0

if puerto_libre(8501):   # solo se inicia si la app no está corriendo ya
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                      "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(8)

output.serve_kernel_port_as_iframe(8501, height=950)   # la app de Streamlit dentro del notebook

**5.3 La misma app desplegada en Streamlit Community Cloud** (enlace público permanente):

In [ ]:
from IPython.display import IFrame
IFrame("https://prediccion-acv-upb.streamlit.app/?embed=true", width="100%", height=950)

## 6. Pantallazo del despliegue

Interfaz de Streamlit desplegada en https://prediccion-acv-upb.streamlit.app con el paciente 0 de `datos_futuros.csv` (probabilidad de ACV 67,8 %, riesgo ALTO):

(pegar aquí el pantallazo: arrastrar la imagen a esta celda de texto)

## 7. Conclusión del despliegue

- El modelo final (**Random Forest optimizado con GridSearch**) se desplegó con una **interfaz gráfica web hecha en Streamlit**, publicada en
  **Streamlit Community Cloud** desde un repositorio público de GitHub: https://prediccion-acv-upb.streamlit.app. Cualquier persona la
  puede usar sin saber Python.
- Como el `.pkl` guarda el **pipeline completo**, los datos nuevos reciben automáticamente el mismo escalado del entrenamiento. Esto
  corrige el error de la práctica anterior, donde el modelo siempre predecía 0.
- Antes de predecir se **validan los datos de entrada**. Por ejemplo, se detectó y corrigió la fila de `datos_futuros.csv` con dos tipos
  de trabajo activos.
- La interfaz muestra la **probabilidad** y el **nivel de riesgo**, no solo la clase. Como el modelo tiene buen recall pero precisión baja,
  sirve como **herramienta de alerta (tamizaje)**, no como diagnóstico.